# 01: Introduction to Finetuning in 2026

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/finetuning/01_intro_to_finetuning_2026.ipynb)

Finetuning changes the model itself. That makes it powerful, but it also makes it expensive, sticky, and easy to misuse. This notebook explains where finetuning belongs in a modern stack that already includes prompting, retrieval, tools, agents, and evaluation.

The guiding idea is simple: change the model only after cheaper levers have been used well and measured honestly.

## 🎯 Learning Objectives

By the end of this notebook, you will:

- Understand **Learning goals**
- Understand **Why another lever exists at all**
- Understand **A practical escalation path**
- Understand **When finetuning helps**
- Understand **When finetuning is the wrong tool**


In [ ]:
# --- Google Colab Pro Fine-Tuning + Evaluation Setup ---
%%capture
!pip install -q --upgrade pip
!pip install -q unsloth datasets trl peft accelerate bitsandbytes pandas matplotlib scikit-learn

import json
import math
import random
import time
from pathlib import Path
from typing import Any, Dict, List

import torch
from datasets import Dataset, DatasetDict, load_dataset
from google.colab import drive
from transformers import TrainingArguments
from trl import SFTTrainer
from unsloth import FastLanguageModel

drive.mount('/content/drive')

CACHE_DIR = "/content/drive/MyDrive/models"
OUTPUT_DIR = "/content/drive/MyDrive/castalia-finetuning"
MODEL_NAME = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 2048
DTYPE = None
LOAD_IN_4BIT = True

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
    cache_dir=CACHE_DIR,
)

tokenizer.padding_side = "right"

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

def formatting_prompts_func(batch):
    return {"text": batch["text"]}

print("✓ Fine-tuning stack ready")
print("  Model:", MODEL_NAME)
print("  CUDA device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("  BF16 supported:", torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 120)

## Learning goals

- understand when finetuning helps and when prompting, RAG, or agents are better
- distinguish supervised finetuning, preference tuning, and RL style post-training
- adopt an adapter-first doctrine for Colab-realistic work
- see why evaluation should happen before training, not after
- build a practical decision rubric you can reuse on future projects

In [ ]:
import math
import random
import statistics
from pathlib import Path

random.seed(2026)

ARTIFACT_DIR = Path("artifacts") / "notebook_01_intro_to_finetuning"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

def print_table(rows, headers=None):
    rows = list(rows)
    if not rows:
        print("No rows")
        return

    if headers is None:
        if isinstance(rows[0], dict):
            headers = list(rows[0].keys())
        else:
            headers = ["col_" + str(i) for i in range(len(rows[0]))]

    normalized = []
    for row in rows:
        if isinstance(row, dict):
            normalized.append([str(row.get(header, "")) for header in headers])
        else:
            normalized.append([str(value) for value in row])

    widths = []
    for index, header in enumerate(headers):
        content_width = max(len(values[index]) for values in normalized)
        widths.append(max(len(str(header)), content_width))

    header_line = " | ".join(str(header).ljust(widths[index]) for index, header in enumerate(headers))
    divider = "-+-".join("-" * width for width in widths)
    print(header_line)
    print(divider)
    for values in normalized:
        print(" | ".join(values[index].ljust(widths[index]) for index in range(len(headers))))

def clipped(value, low=0.0, high=1.0):
    return max(low, min(high, value))

print("Artifact directory:", ARTIFACT_DIR.resolve())

## Why another lever exists at all

You already have four strong levers before finetuning:

1. prompting changes instructions
2. RAG changes what evidence is present
3. agents change how work is decomposed and which tools are used
4. evals change whether you can tell if anything improved

Finetuning is different because it changes the model's learned behavior. That is useful when the behavior should be repeatedly available without re-explaining it every time.

In [ ]:
capability_map = [
    {
        "lever": "Prompting",
        "best_for": "Instruction clarity and decomposition",
        "latency_cost": "lowest",
        "data_needed": "none",
        "changes_model": "no",
    },
    {
        "lever": "RAG",
        "best_for": "Knowledge grounding and evidence injection",
        "latency_cost": "low_to_medium",
        "data_needed": "documents not labels",
        "changes_model": "no",
    },
    {
        "lever": "Agents",
        "best_for": "Tool use and multi-step workflows",
        "latency_cost": "medium_to_high",
        "data_needed": "tool contracts and prompts",
        "changes_model": "no",
    },
    {
        "lever": "Finetuning",
        "best_for": "Stable behavior change at generation time",
        "latency_cost": "training upfront, often faster inference behavior",
        "data_needed": "labeled examples or preferences",
        "changes_model": "yes",
    },
]

print_table(capability_map, headers=["lever", "best_for", "latency_cost", "data_needed", "changes_model"])

## A practical escalation path

In real projects, the order should usually be:

1. fix the prompt
2. add grounding if the problem is missing knowledge
3. add tools or workflow structure if the problem is action or decomposition
4. evaluate the baseline
5. only then ask whether the model itself should change

That ordering keeps you from training a model to compensate for a system design problem.

In [ ]:
scenarios = [
    {
        "task": "Answer policy questions with citations from a handbook",
        "behavior_gap": 0.30,
        "knowledge_gap": 0.85,
        "tool_gap": 0.15,
        "consistency_need": 0.35,
    },
    {
        "task": "Generate customer support replies in a house style",
        "behavior_gap": 0.82,
        "knowledge_gap": 0.20,
        "tool_gap": 0.10,
        "consistency_need": 0.88,
    },
    {
        "task": "Reconcile numbers from an internal database before answering",
        "behavior_gap": 0.35,
        "knowledge_gap": 0.30,
        "tool_gap": 0.92,
        "consistency_need": 0.60,
    },
    {
        "task": "Write short SQL with a strict organization style guide",
        "behavior_gap": 0.75,
        "knowledge_gap": 0.10,
        "tool_gap": 0.20,
        "consistency_need": 0.82,
    },
]

def choose_first_lever(row):
    scores = {
        "Prompting": 0.55 * row["behavior_gap"] + 0.20 * row["consistency_need"],
        "RAG": 0.90 * row["knowledge_gap"],
        "Agents": 0.95 * row["tool_gap"],
        "Finetuning": 0.75 * row["behavior_gap"] + 0.60 * row["consistency_need"] - 0.35 * row["knowledge_gap"],
    }
    best = max(scores, key=scores.get)
    return best, scores

rows = []
for row in scenarios:
    best, scores = choose_first_lever(row)
    rows.append({
        "task": row["task"],
        "recommended_first_move": best,
        "prompt_score": round(scores["Prompting"], 2),
        "rag_score": round(scores["RAG"], 2),
        "agent_score": round(scores["Agents"], 2),
        "finetune_score": round(scores["Finetuning"], 2),
    })

print_table(rows, headers=["task", "recommended_first_move", "prompt_score", "rag_score", "agent_score", "finetune_score"])

## When finetuning helps

Finetuning is strongest when the task requires repeated behavior change, not just one-off instruction following. Examples:

- a durable output style or voice
- consistent schema habits
- domain-specific decision boundaries
- improved tool or format priors
- shorter prompts because the behavior is baked in

The key word is repeated. If the same behavior matters thousands of times, training can amortize the upfront cost.

In [ ]:
projects = [
    {
        "project": "Medical scribe formatting",
        "request_volume": 20000,
        "behavior_repeatability": 0.92,
        "knowledge_volatility": 0.25,
        "eval_readiness": 0.90,
    },
    {
        "project": "FAQ assistant on fast-changing docs",
        "request_volume": 9000,
        "behavior_repeatability": 0.35,
        "knowledge_volatility": 0.95,
        "eval_readiness": 0.78,
    },
    {
        "project": "Structured case-note summarizer",
        "request_volume": 12000,
        "behavior_repeatability": 0.86,
        "knowledge_volatility": 0.30,
        "eval_readiness": 0.84,
    },
    {
        "project": "Spreadsheet analyst with live database lookups",
        "request_volume": 6000,
        "behavior_repeatability": 0.52,
        "knowledge_volatility": 0.65,
        "eval_readiness": 0.66,
    },
]

def finetune_readiness(project):
    volume_score = clipped(math.log10(project["request_volume"]) / 5.0)
    score = (
        0.30 * project["behavior_repeatability"]
        + 0.25 * volume_score
        + 0.25 * project["eval_readiness"]
        - 0.20 * project["knowledge_volatility"]
    )
    return round(clipped(score), 3)

for project in projects:
    project["finetune_readiness"] = finetune_readiness(project)
    project["likely_move"] = "train_adapter_first" if project["finetune_readiness"] >= 0.55 else "improve_system_first"

print_table(
    projects,
    headers=["project", "request_volume", "behavior_repeatability", "knowledge_volatility", "eval_readiness", "finetune_readiness", "likely_move"],
)

## When finetuning is the wrong tool

Many teams reach for training when the real problem is one of these:

- missing evidence
- poor evaluation
- bad workflow design
- unclear task definitions
- tiny datasets with weak labels

Training a model on noisy goals can make the system look more committed while still being wrong.

In [ ]:
anti_patterns = [
    {
        "situation": "The model hallucinates facts from changing company docs",
        "better_fix": "RAG plus citation evals",
        "why_not_train": "weights will age faster than the docs",
    },
    {
        "situation": "The agent picks the wrong tool",
        "better_fix": "better routing prompts and tool contracts",
        "why_not_train": "the failure is orchestration, not knowledge",
    },
    {
        "situation": "Stakeholders cannot agree on what good looks like",
        "better_fix": "rubrics and eval set design",
        "why_not_train": "training before targets are clear locks in confusion",
    },
    {
        "situation": "You have 60 perfect examples and 6000 edge cases",
        "better_fix": "collect more representative data",
        "why_not_train": "the adapter will memorize a narrow slice",
    },
]

print_table(anti_patterns, headers=["situation", "better_fix", "why_not_train"])

## The post-training ladder

The modern open-source ladder usually looks like this:

1. supervised finetuning teaches a target behavior directly
2. preference tuning nudges the model toward better choices among plausible outputs
3. RL style methods help when you have strong reward signals or verifiable tasks

This notebook is about deciding when to enter that ladder at all.

In [ ]:
training_stages = [
    {
        "stage": "SFT",
        "core_signal": "demonstrations",
        "question_answered": "What should a good output look like?",
        "default_use": "first training stage",
        "colab_realism": "high",
    },
    {
        "stage": "Preference tuning",
        "core_signal": "chosen vs rejected outputs or desirability labels",
        "question_answered": "Which acceptable output is better?",
        "default_use": "after SFT baseline exists",
        "colab_realism": "medium",
    },
    {
        "stage": "RL style post-training",
        "core_signal": "reward from a verifier, environment, or judge",
        "question_answered": "Can the model improve through trial and feedback?",
        "default_use": "only on tasks with useful rewards",
        "colab_realism": "limited but possible for small experiments",
    },
]

print_table(training_stages, headers=["stage", "core_signal", "question_answered", "default_use", "colab_realism"])

## First principles for SFT, preferences, and RL

These stages solve different problems:

- SFT is imitation of desired outputs
- preference tuning is ranking among outputs
- RL is optimization against a reward signal over trajectories or outcomes

If you do not know what signal you actually have, you do not yet know which training stage fits.

In [ ]:
dataset_inventory = [
    {"dataset": "20000 clean input-output pairs", "best_stage": "SFT"},
    {"dataset": "5000 chosen-rejected pairs", "best_stage": "Preference tuning"},
    {"dataset": "unit tests and executable verifier", "best_stage": "RL style post-training"},
    {"dataset": "raw PDFs with no labels", "best_stage": "Not training yet"},
]

def explain_stage_choice(name):
    if name == "SFT":
        return "Direct demonstrations are present."
    if name == "Preference tuning":
        return "Relative quality judgments are present."
    if name == "RL style post-training":
        return "A reward or verifier can score behavior."
    return "You need formatting, retrieval, or labeling first."

rows = []
for item in dataset_inventory:
    rows.append({
        "dataset": item["dataset"],
        "best_stage": item["best_stage"],
        "reason": explain_stage_choice(item["best_stage"]),
    })

print_table(rows, headers=["dataset", "best_stage", "reason"])

## Adapter-first doctrine

For most teams, especially on Colab Pro, dense finetuning should not be the default. Adapters win because they are:

- cheaper to train
- faster to iterate
- easier to compare
- easier to roll back
- easier to store and share

That is why this course treats adapter tuning as the normal path.

In [ ]:
model_sizes = [
    {"model": "3B", "params_billions": 3.0},
    {"model": "4B", "params_billions": 4.0},
    {"model": "8B", "params_billions": 8.0},
]

def dense_checkpoint_gb(params_billions, bytes_per_param=2.0):
    return params_billions * 1_000_000_000 * bytes_per_param / (1024 ** 3)

def adapter_checkpoint_mb(params_billions, tuned_fraction=0.002, bytes_per_param=2.0):
    tuned_params = params_billions * 1_000_000_000 * tuned_fraction
    return tuned_params * bytes_per_param / (1024 ** 2)

rows = []
for config in model_sizes:
    dense_gb = dense_checkpoint_gb(config["params_billions"])
    adapter_mb = adapter_checkpoint_mb(config["params_billions"])
    rows.append({
        "model": config["model"],
        "dense_checkpoint_gb": round(dense_gb, 2),
        "adapter_checkpoint_mb": round(adapter_mb, 1),
        "size_ratio": round((dense_gb * 1024) / adapter_mb, 1),
    })

print_table(rows, headers=["model", "dense_checkpoint_gb", "adapter_checkpoint_mb", "size_ratio"])

## Why storage and rollback matter

Training is not just optimization. It is experiment management.

If you can save ten adapters instead of ten full checkpoints, you can:

- compare variants directly
- keep failed runs for inspection
- merge or discard later
- ship a small artifact instead of a huge one

That operational flexibility is one reason adapter-first practice survived beyond the first LoRA wave.

In [ ]:
experiment_options = [
    {"option": "Dense run", "train_cost": 0.95, "rollback_ease": 0.20, "shareability": 0.25},
    {"option": "LoRA run", "train_cost": 0.50, "rollback_ease": 0.82, "shareability": 0.86},
    {"option": "QLoRA run", "train_cost": 0.35, "rollback_ease": 0.88, "shareability": 0.90},
]

for option in experiment_options:
    option["ops_score"] = round(
        0.40 * (1 - option["train_cost"])
        + 0.30 * option["rollback_ease"]
        + 0.30 * option["shareability"],
        3,
    )

print_table(experiment_options, headers=["option", "train_cost", "rollback_ease", "shareability", "ops_score"])

## Evaluation-first before training

If you train before building an eval set, you cannot answer the most important question:

Did the model get better on the thing we actually care about?

Evaluation-first means:

1. define tasks
2. build a baseline
3. label representative examples
4. decide pass and fail rules
5. then train against that target

In [ ]:
baseline_eval = [
    {"bucket": "format_adherence", "base_score": 0.62, "target_score": 0.88, "examples": 140},
    {"bucket": "domain_terminology", "base_score": 0.57, "target_score": 0.82, "examples": 120},
    {"bucket": "citation_correctness", "base_score": 0.41, "target_score": 0.80, "examples": 110},
    {"bucket": "tool_call_accuracy", "base_score": 0.76, "target_score": 0.90, "examples": 90},
]

for row in baseline_eval:
    row["gap"] = round(row["target_score"] - row["base_score"], 2)
    row["priority"] = "trainable_behavior" if row["bucket"] != "citation_correctness" else "maybe_rag_not_training"

print_table(baseline_eval, headers=["bucket", "base_score", "target_score", "gap", "examples", "priority"])

## Why this gate protects you

Notice the bucket called citation correctness. If that gap is mainly due to missing evidence, training may be the wrong fix. The baseline tells you where the weakness lives:

- in the prompt
- in the retrieval layer
- in the tool path
- or in the model's learned behavior

That diagnosis should happen before you spend GPU hours.

In [ ]:
improvement_options = [
    {"bucket": "format_adherence", "prompt_gain": 0.08, "rag_gain": 0.00, "agent_gain": 0.02, "train_gain": 0.20},
    {"bucket": "domain_terminology", "prompt_gain": 0.05, "rag_gain": 0.06, "agent_gain": 0.01, "train_gain": 0.18},
    {"bucket": "citation_correctness", "prompt_gain": 0.03, "rag_gain": 0.28, "agent_gain": 0.04, "train_gain": 0.07},
    {"bucket": "tool_call_accuracy", "prompt_gain": 0.06, "rag_gain": 0.00, "agent_gain": 0.19, "train_gain": 0.08},
]

rows = []
for bucket in improvement_options:
    moves = {
        "prompting": bucket["prompt_gain"],
        "rag": bucket["rag_gain"],
        "agents": bucket["agent_gain"],
        "finetuning": bucket["train_gain"],
    }
    best_move = max(moves, key=moves.get)
    rows.append({
        "bucket": bucket["bucket"],
        "best_move": best_move,
        "expected_gain": round(moves[best_move], 2),
    })

print_table(rows, headers=["bucket", "best_move", "expected_gain"])

## Comparison matrix: prompting vs RAG vs agents vs finetuning

No single lever dominates all tasks. The better question is which lever best addresses the current bottleneck.

In [ ]:
comparison_matrix = [
    {
        "criterion": "Fixes missing factual knowledge",
        "prompting": 1,
        "rag": 5,
        "agents": 3,
        "finetuning": 2,
    },
    {
        "criterion": "Improves repeated style and structure",
        "prompting": 3,
        "rag": 1,
        "agents": 2,
        "finetuning": 5,
    },
    {
        "criterion": "Adds tool use and external actions",
        "prompting": 2,
        "rag": 1,
        "agents": 5,
        "finetuning": 2,
    },
    {
        "criterion": "Fastest to test",
        "prompting": 5,
        "rag": 4,
        "agents": 3,
        "finetuning": 1,
    },
    {
        "criterion": "Most stable once trained well",
        "prompting": 2,
        "rag": 3,
        "agents": 3,
        "finetuning": 5,
    },
]

print_table(
    comparison_matrix,
    headers=["criterion", "prompting", "rag", "agents", "finetuning"],
)

## Build a reusable decision rubric

A small rubric is more useful than a vague intuition. We can score whether a project looks like a training problem or a system-design problem.

In [ ]:
decision_inputs = [
    {
        "name": "Brand-safe customer reply assistant",
        "behavior_drift": 0.88,
        "knowledge_change_rate": 0.25,
        "tool_dependence": 0.20,
        "label_quality": 0.91,
        "eval_quality": 0.89,
    },
    {
        "name": "Policy chatbot over a live handbook",
        "behavior_drift": 0.30,
        "knowledge_change_rate": 0.92,
        "tool_dependence": 0.10,
        "label_quality": 0.64,
        "eval_quality": 0.85,
    },
    {
        "name": "Invoice operations copilot with database actions",
        "behavior_drift": 0.52,
        "knowledge_change_rate": 0.40,
        "tool_dependence": 0.87,
        "label_quality": 0.72,
        "eval_quality": 0.80,
    },
]

def choose_strategy(case):
    finetune_score = (
        0.35 * case["behavior_drift"]
        + 0.25 * case["label_quality"]
        + 0.25 * case["eval_quality"]
        - 0.20 * case["knowledge_change_rate"]
        - 0.10 * case["tool_dependence"]
    )
    rag_score = 0.65 * case["knowledge_change_rate"] + 0.10 * case["eval_quality"]
    agent_score = 0.70 * case["tool_dependence"] + 0.15 * case["behavior_drift"]
    prompt_score = 0.35 * case["behavior_drift"] + 0.10 * case["eval_quality"]

    scores = {
        "prompting": round(clipped(prompt_score), 3),
        "rag": round(clipped(rag_score), 3),
        "agents": round(clipped(agent_score), 3),
        "finetuning": round(clipped(finetune_score), 3),
    }
    best = max(scores, key=scores.get)
    return best, scores

rows = []
for case in decision_inputs:
    best, scores = choose_strategy(case)
    rows.append({
        "name": case["name"],
        "best_strategy": best,
        "prompting": scores["prompting"],
        "rag": scores["rag"],
        "agents": scores["agents"],
        "finetuning": scores["finetuning"],
    })

print_table(rows, headers=["name", "best_strategy", "prompting", "rag", "agents", "finetuning"])

## Takeaways

- finetuning is a behavior-change lever, not a universal fix
- SFT is usually first, preference tuning second, RL later
- adapters should be your default starting point on Colab-class hardware
- evaluation should exist before the first training run
- the best teams ask why the model is failing before they ask how to train it

In the next notebook, we will make the environment concrete by setting up Colab Pro and Unsloth for a first dry run.

## 🏋️ Exercises

Put your understanding to the test:

**Exercise 1 — Explore:** modify a hyperparameter and track its effect on loss curves. Document what changes and why.

**Exercise 2 — Build:** prepare a dataset from a new domain using the techniques shown. Compare your implementation to the one in this notebook.

**Exercise 3 — Extend:** compare the finetuned model against the base model on 5 test prompts.